# <center> XGBoost + Isolation Forest (Confidence-Gated Hybrid)

Hybrid improves robustness by adding anomaly verification only for uncertain decisions.

Use anomaly detector only when XGBoost is uncertain.

In [3]:
pip install XGboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

In [6]:
df = pd.read_csv("train_test_network.csv")
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [7]:
df = df.sample(30000, random_state=42)

In [8]:
X = df.drop(["label","type"],axis=1)
y = df["label"]

In [9]:
le = LabelEncoder()
for col in X.columns:
    if X[col].dtype=="object":
        X[col]=le.fit_transform(X[col].astype(str))

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42)

In [11]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [12]:
xgb = XGBClassifier(n_estimators=200,max_depth=8,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,random_state=42)

In [16]:
X_normal = X_train[y_train==0]
iso = IsolationForest(n_estimators=200,contamination=0.30,random_state=42)

In [17]:
start_train=time.time()
xgb.fit(X_train,y_train)
X_normal=X_train[y_train==0]
iso.fit(X_normal)
print("Training Time:",time.time()-start_train)

Training Time: 1.1498653888702393


In [18]:
start_pred=time.time()
probs=xgb.predict_proba(X_test)[:,1]
hybrid_preds=[]
for i,row in enumerate(X_test):
    # confident attack
    if probs[i] > 0.70:
        hybrid_preds.append(1)
    # confident normal
    elif probs[i] < 0.30:
        hybrid_preds.append(0)
    # uncertain region → consult anomaly detector
    else:
        anomaly=iso.predict(row.reshape(1,-1))[0]
        if anomaly==-1:
            hybrid_preds.append(1)
        else:
            hybrid_preds.append(0)

hybrid_preds=np.array(hybrid_preds)
print("Prediction Time:",time.time()-start_pred)

Prediction Time: 0.04237556457519531


In [19]:
print("Hybrid Accuracy:",accuracy_score(y_test,hybrid_preds))
print(classification_report(y_test,hybrid_preds))

Hybrid Accuracy: 0.9997777777777778
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2106
           1       1.00      1.00      1.00      6894

    accuracy                           1.00      9000
   macro avg       1.00      1.00      1.00      9000
weighted avg       1.00      1.00      1.00      9000



In [ ]:
# best_model=